<a href="https://www.kaggle.com/code/avikdas567/time-series-forecasting-with-lag-features-lightgbm?scriptVersionId=306836925" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import gc
import numpy as np
import pandas as pd

import lightgbm as lgb

# PATHS

TRAIN_PATH = "/kaggle/input/competitions/ts-forecasting/train.parquet"
TEST_PATH  = "/kaggle/input/competitions/ts-forecasting/test.parquet"

# LOAD DATA

print("Loading data...")
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

test_original = test.copy()

# SETUP

ID_COL = "id"
TIME_COL = "ts_index"

FEATURES = [c for c in train.columns if c.startswith("feature_")]

# Detect target
KNOWN_COLS = set([ID_COL, TIME_COL, "code", "sub_code", "sub_category", "horizon", "weight"] + FEATURES)
TARGET = [c for c in train.columns if c not in KNOWN_COLS][0]

print("TARGET:", TARGET)

# SORT

train = train.sort_values(TIME_COL).reset_index(drop=True)
test  = test.sort_values(TIME_COL).reset_index(drop=True)

# FAST FEATURES

print("Creating simple features...")

train["lag_1"] = train[TARGET].shift(1)

train = train.dropna().reset_index(drop=True)

ALL_FEATURES = FEATURES + ["lag_1"]

# TRAIN

print("Training...")

model = lgb.LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=31,
    n_estimators=200,
    verbosity=-1
)

model.fit(train[ALL_FEATURES], train[TARGET], sample_weight=train["weight"])

# TEST FEATURES

print("Processing test...")

test["lag_1"] = 0

# PREDICTION

print("Predicting...")
test_preds = model.predict(test[ALL_FEATURES])

# SUBMISSION

submission = pd.DataFrame({
    "id": test_original["id"],
    "prediction": test_preds
})

submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created!")
print(submission.head())

Loading data...
TARGET: y_target
Creating simple features...
Training...
Processing test...
Predicting...

'submission.csv' created!
                                       id  prediction
0   W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647   -0.710092
1  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647   -0.743840
2  W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647    0.000830
3   W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647    0.000791
4  W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648   -0.003795
